## The queue: `noise` → `ids_shuffle` → `ids_restore`
The same lines as in `random_masking` (models_mae.py), run on a toy of 6 tokens, keeping 2:
- **`ids_shuffle`** = the random **queue**: who stands at queue spot *k*
- **`ids_restore`** = each token's **place in that queue**


In [ ]:
import torch
import matplotlib.pyplot as plt

SEED = 3   # change for a different random queue


In [ ]:
letters = list("ABCDEF")
L, len_keep = 6, 2
torch.manual_seed(SEED)
noise = torch.rand(1, L)                              # one random ticket per token
ids_shuffle = torch.argsort(noise, dim=1)             # the queue: token numbers, smallest ticket first
ids_restore = torch.argsort(ids_shuffle, dim=1)       # each token's spot in the queue

print("token       :", "   ".join(f"{c:>4}" for c in letters))
print("noise       :", "   ".join(f"{v:4.2f}" for v in noise[0].tolist()))
print()
print("ids_shuffle :", ids_shuffle[0].tolist(), "-> queue:", [letters[i] for i in ids_shuffle[0].tolist()])
print("ids_restore :", ids_restore[0].tolist(), "-> A is at spot", ids_restore[0, 0].item(),
      ", B at spot", ids_restore[0, 1].item(), ", ...")

In [ ]:
KEEP, HIDE, GREY = "#3aa6a0", "#e07a5f", "#dddddd"
fig, ax = plt.subplots(figsize=(9, 4.2))
def box(x, y, label, color, sub=None):
    ax.add_patch(plt.Rectangle((x - 0.4, y - 0.3), 0.8, 0.6, color=color))
    ax.text(x, y, label, ha="center", va="center", fontsize=14, color="white")
    if sub: ax.text(x, y - 0.5, sub, ha="center", va="center", fontsize=9, color="#555")

y_tok, y_q, y_back = 3, 1.3, -0.4
spot_of = ids_restore[0].tolist()                    # token i -> queue spot

# row 1: original tokens with their noise
for i, c in enumerate(letters):
    box(i, y_tok, c, KEEP if spot_of[i] < len_keep else HIDE, f"noise {noise[0, i]:.2f}")
ax.text(-1, y_tok, "tokens (original)", ha="right", va="center", fontsize=10)

# row 2: the queue = ids_shuffle
for k, i in enumerate(ids_shuffle[0].tolist()):
    box(k, y_q, letters[i], KEEP if k < len_keep else HIDE, f"spot {k}")
ax.text(-1, y_q, "ids_shuffle (queue)", ha="right", va="center", fontsize=10)
ax.axvline(len_keep - 0.5, ymin=0.38, ymax=0.62, color="black", ls="--")
ax.text(len_keep - 0.5, y_q + 0.55, "keep | hide", ha="center", fontsize=9)

# arrows: token i -> its queue spot  (this mapping IS ids_restore)
for i in range(L):
    ax.annotate("", xy=(spot_of[i], y_q + 0.32), xytext=(i, y_tok - 0.62),
                arrowprops=dict(arrowstyle="->", color="#888", lw=1.2))

# row 3: ids_restore written under each token
for i, c in enumerate(letters):
    ax.text(i, y_back, f"{c}→{spot_of[i]}", ha="center", fontsize=12)
ax.text(-1, y_back, "ids_restore (token → its spot)", ha="right", va="center", fontsize=10)

ax.set_xlim(-3.6, L); ax.set_ylim(-1, 3.8); ax.axis("off")
ax.set_title("Arrows = ids_restore: where each token went in the queue. Teal = kept, orange = hidden")
plt.show()